# Golden shiners: train a point detector, then track with TREx

A complete loop on one downloadable dataset -- **published tracks -> pseudo-annotations ->
a point model -> TREx tracking -> an annotated video**. Nothing is labelled by hand.

The dataset ships three concatenated recordings (10, 30 and 70 fish), 80 k-means-selected
frames from each, and the published SchoolTracker tables. Everything below is rebuilt from
those three things.

## Two ways to give TREx a body

TREx fits a midline through an animal's **outline**, so the interesting question is always
where the outline comes from. This notebook runs both answers:

- **Route A -- a trained point model.** POLO detects a point per animal. TREx wraps each
  detection in a disc and recovers the outline from the pixels inside it.
- **Route B -- no model at all.** TREx's own background subtraction: the blob *is* the
  animal, outline included.

Route B needs no training and runs out of the box, so it is the default here. Route A is
the reason the annotation and training sections exist, and switches on the moment you have
a model.

**No model weights ship with this dataset** -- section 3 says why -- so on the default
settings route A is skipped and only route B runs. Sections 5-8 then describe one route,
not two; train a model, or point `POLO_MODEL` at your own, and the comparison appears.

**The data** is [EcodylicScience/mosaic-example-shiners-polo][card] on Hugging Face
(~1.05 GB), fetched by the next cell.

[card]: https://huggingface.co/datasets/EcodylicScience/mosaic-example-shiners-polo

## What you need

mosaic, plus [TREx](https://trex.run) in its own environment for sections 4 onward.
Training in section 3 additionally needs a POLO environment; without one the notebook
prints the command to run there and carries on.

## The data

Free-swimming golden shiners (*Notemigonus crysoleucas*) in a 2.1 x 1.2 m tank, filmed from
above. One trial at each of three group sizes, ~24,600 frames each.

**Citation.** Davidson JD, Sosna MMG, Twomey CR, Sridhar VH, Leblanc SP, Couzin ID (2021)
Collective detection based on visual information in animal groups. *Journal of the Royal
Society Interface* 18: 20210142.
[doi:10.1098/rsif.2021.0142](https://doi.org/10.1098/rsif.2021.0142)

**Data licence.** CC0 1.0, from Dryad:
[doi:10.5061/dryad.sbcc2fr2h](https://doi.org/10.5061/dryad.sbcc2fr2h). The re-encoded
videos, extracted frames and converted tables on Hugging Face are a derivative of that
record and carry the same licence.

## 0 -- Configuration

In [ ]:
from pathlib import Path
from typing import Optional

# ---- where the data comes from ---------------------------------------------
#
#   "download" -- fetch the dataset from Hugging Face (~1.05 GB). This is what
#                 most people want.
#   "local"    -- a mosaic dataset you already have; set LOCAL_DATASET.
SOURCE = "download"

HF_REPO = "EcodylicScience/mosaic-example-shiners-polo"
HF_ASSET = "shiners-polo.tar.gz"
# Where the archive is unpacked. None -> ./shiners-polo-example
DOWNLOAD_DIR: Optional[Path] = None
# SOURCE="local": the dataset directory (the one holding dataset.yaml).
LOCAL_DATASET: Optional[Path] = None

# ---- where the external tools are ------------------------------------------
# Both are found on the same five-step ladder: the argument, MOSAIC_<TOOL>_BIN,
# MOSAIC_<TOOL>_CONDA_ENV, then $PATH. Naming a conda environment is the reliable
# spelling for TREx, which relaunches itself and resolves its embedded Python
# from CONDA_PREFIX. Empty leaves whatever is already in the environment in
# charge -- do not put your own environment name here, it would override a
# reader's $PATH silently.
TREX_CONDA_ENV = ""
POLO_CONDA_ENV = ""

# ---- section 3: training ----------------------------------------------------
# False by default: no weights ship with this dataset, and training wants a POLO
# environment and a GPU. Set True in one, or point POLO_MODEL at a best.pt you
# already have -- either one switches route A on in section 4.
TRAIN = False
POLO_MODEL: Optional[Path] = None

BASE_MODEL = "polo26n.yaml"
EPOCHS     = 300
IMGSZ      = 1280   # NOT 640: a shiner is ~50 px in a 1920-wide frame, so 640
                    # shrinks it to ~17 px. Drop to 640 only if memory forces it.
BATCH      = 8
DEVICE     = "0"    # CUDA index on a GPU box, "mps" on Apple Silicon, else "cpu"

# ---- the dataset ------------------------------------------------------------
ENTRIES     = [("10-fish", "0066"), ("30-fish", "0084"), ("70-fish", "0103")]
VALID_GROUP = "30-fish"   # held out of training: both density extremes train and
                          # the middle one validates, so the score answers "does
                          # this transfer to a density it never saw"
CLASS_NAME  = "fish"

# ---- section 4: tracking ----------------------------------------------------
TRACK_ENTRIES = ENTRIES[:1]   # just the 10-fish trial; ENTRIES does all three

# TREx converts the whole video before tracking any of it, and detection is the
# expensive half -- roughly 11 frames/s on an M1, so all 24,578 take ~35 minutes.
# A range keeps the loop quick while you are finding your settings; None does the
# lot. It reaches the run identity, so a bounded run and a full one are different
# variants and neither can be mistaken for the other.
CONVERT_FRAMES = 3000        # 100 s at 30 fps -- enough for the clip in section 7

# Route A: what makes a point detection yield a real body axis. See section 4.
DETECT_CONF            = 0.3
BACKGROUND_SUBTRACTION = True
POSTURE_THRESHOLD      = 45

# Route B: TREx's own background subtraction, no model.
BGSUB_DETECT_THRESHOLD = 25
BGSUB_AVERAGING        = "mode"    # per-pixel median; "mean" leaves ghosts of slow fish
BGSUB_AVERAGE_SAMPLES  = 100

print(f"source   {SOURCE}")
print(f"tracking {[f'{g}/{s}' for g, s in TRACK_ENTRIES]}"
      f"{f', first {CONVERT_FRAMES} frames' if CONVERT_FRAMES else ', whole video'}")

In [ ]:
import json
import math
import os
import shutil
import subprocess
import tarfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mosaic.core.dataset import open_dataset
from mosaic.core.helpers import make_entry_key
from mosaic.core.pipeline.index import feature_run_root
from mosaic.core.pipeline.ops import run_op
from mosaic.core.pipeline.tracks_index import read_tracks_index
from mosaic.tracking import get_frame_manifests, list_frame_runs
import mosaic.tracking.ops  # noqa: F401 -- registers train-points / trex for run_op

from mosaic.core.annotations.model import (
    AnnotationFrame,
    AnnotationObject,
    AnnotationSet,
    Keypoint,
    KeypointSchema,
)
from mosaic.tracking.pose_training import make_polo_data_yaml
from mosaic.tracking.pose_training.converters.base import format_polo_label_line
from mosaic.tracking.pose_training.converters.emit import usable_frames, write_split_tree

# ---- resolve DATASET --------------------------------------------------------
if SOURCE == "download":
    try:
        from huggingface_hub import hf_hub_download
    except ImportError as exc:  # pragma: no cover - environment guidance
        raise ImportError(
            "SOURCE='download' needs huggingface_hub:\n"
            "    pip install 'huggingface_hub>=1.2.0'\n"
            "The version floor matters: older clients retry a rate-limit "
            "response with a 25-second backoff against a 5-minute window, so "
            "they fail rather than wait. Set SOURCE='local' to point at a copy "
            "you already have."
        ) from exc

    root = Path(DOWNLOAD_DIR) if DOWNLOAD_DIR else Path.cwd() / "shiners-polo-example"
    root.mkdir(parents=True, exist_ok=True)
    DATASET = root / "shiners-polo"
    if DATASET.exists():
        print(f"have      {DATASET}")
    else:
        print(f"fetching  {HF_ASSET} (~1.05 GB) ...")
        archive = hf_hub_download(HF_REPO, HF_ASSET, repo_type="dataset")
        with tarfile.open(archive) as tar:
            tar.extractall(root, filter="data")

elif SOURCE == "local":
    if LOCAL_DATASET is None:
        raise ValueError("SOURCE='local' needs LOCAL_DATASET set to a mosaic dataset "
                         "directory (the one holding dataset.yaml).")
    DATASET = Path(LOCAL_DATASET)

else:
    raise ValueError(f"SOURCE must be 'download' or 'local', not {SOURCE!r}")

assert (DATASET / "dataset.yaml").exists(), f"no dataset.yaml under {DATASET}"
ds = open_dataset(DATASET)
print(f"dataset   {DATASET}")
print(f"          {ds.name}, manifest v{ds.manifest.manifest_version}")

## 1 -- What is in the dataset

Three roots, and section 2 rebuilds the training set from the first two:

- `tracks/` -- the published SchoolTracker tables, which become the labels
- `media/frames/` -- 80 k-means-selected frames per recording, the images a model trains on
- `media_raw/` -- the three concatenated recordings, which TREx tracks in section 4

The POLO training tree itself is **not** shipped. It is a rearrangement of the frames and
the tables -- the same pixels under different filenames -- so shipping it would enlarge the
download to carry nothing new. Rebuilding it takes seconds and is also the part of the
recipe you would want to change.

In [ ]:
# match_media_rows is the typed read: it keeps group/sequence as text (a sequence
# named "0066" is a name, not the number 66) while fps and frame_count stay numeric.
media = pd.concat([ds.match_media_rows(g, s) for g, s in ENTRIES], ignore_index=True)
print(f"media_raw: {len(media)} recordings")
print(media[["group", "sequence", "width", "height", "fps", "frame_count"]]
      .to_string(index=False))

# Resolved from the index rather than pasted here, so this keeps working if the
# archive is rebuilt under a new digest.
PUBLISHED_PRODUCER = "convert-schooltracker_fov_h5"
tracks_idx = read_tracks_index(ds)
published = tracks_idx[tracks_idx["producer"] == PUBLISHED_PRODUCER]
assert len(published), f"no {PUBLISHED_PRODUCER} tables in this dataset"
variants = published["run_id"].unique()
assert len(variants) == 1, f"expected one published variant, found {list(variants)}"
VARIANT = str(variants[0])

print(f"\npublished tracks variant: {VARIANT}")
print(published[["group", "sequence", "n_rows", "frame_min", "frame_max"]]
      .to_string(index=False))

runs = list_frame_runs(ds, method="kmeans")
FRAME_RUN = str(runs.iloc[-1]["run_id"])
manifests = get_frame_manifests(ds, "kmeans", run_id=FRAME_RUN)
print(f"\nframe run {FRAME_RUN}: {len(manifests)} sequences, "
      f"{sum(len(m['files']) for m in manifests)} images")

## 2 -- Rebuild the POLO training tree

A POLO label row is `<class_id> <radius> <x_rel> <y_rel>` -- one point per animal, with a
radius that tells the trainer how much of the image around that point is the object.

**Which point matters, and this one is the head.** The published `X`/`Y` in this format is
the head, not the body centre, so the model below learns to detect heads -- and a tracker
driven by it reports head positions in a column mosaic's schema says is the body centre.
Section 5 measures that and says what to do about it. Keeping the head here is deliberate:
it is what the source data offers directly, and the mismatch is more useful shown than
quietly avoided.

The radius is derived from the data rather than typed in. `body_length` in this format runs
about 1.5x the true fish, so the measured median divides down to a real length, and a third
of that is a head-sized region.

In [ ]:
lengths = pd.concat([ds.load_tracks(g, s, run_id=VARIANT)["body_length"] for g, s in ENTRIES])
median_bl = float(lengths.median())
RADIUS_PX = round(median_bl / 1.5 / 3, 1)
print(f"median body_length {median_bl:.1f} px -> fish ~{median_bl / 1.5:.0f} px "
      f"-> RADIUS_PX = {RADIUS_PX}")

Now the annotations. mosaic has `tracks_to_yolo_pose`, but that emits *pose* labels, and
the `convert-points` op reads CVAT XML -- so there is no built-in "tracks to POLO points".
Building an `AnnotationSet` by hand and emitting through mosaic's own split-tree writer is
the part of this notebook worth reusing for another point model.

In [ ]:
SCHEMA = KeypointSchema(names=("head",))

# A manifest names its entry by its own directory, not by a group/sequence field, and
# `files` carries a (path, frame_index, width, height) record per image. The frame
# index is what joins an image back to the rows that describe it.
entry_of = {make_entry_key(g, s): (g, s) for g, s in ENTRIES}

frames: list[AnnotationFrame] = []
for manifest in manifests:
    key = Path(str(manifest["output_dir"])).name
    group, sequence = entry_of[key]
    table = ds.load_tracks(group, sequence, run_id=VARIANT)
    n_fish = int(table["id"].nunique())
    by_frame = {int(f): sub for f, sub in table.groupby("frame")}

    for record in manifest["files"]:
        frame_no = int(record["frame_index"])
        rows = by_frame.get(frame_no)
        # A frame missing an animal teaches the detector that a real fish is
        # background, which is worse than having one frame fewer. Refuse rather than
        # quietly train on it.
        assert rows is not None and len(rows) == n_fish, (
            f"{key} frame {frame_no}: {0 if rows is None else len(rows)} of {n_fish} fish"
        )
        objects = tuple(
            AnnotationObject(
                keypoints=(Keypoint(x=float(r.X), y=float(r.Y), visibility=2),),
                category=CLASS_NAME,
                track_id=str(int(r.id)),
            )
            for r in rows.itertuples()
        )
        # Renamed: all three recordings contain frame_000123.png, and the stems would
        # collide once the splits merge into one tree. An AnnotationFrame is named by
        # its path, so the rename has to happen on disk -- staged just below.
        frames.append(AnnotationFrame(
            image_path=Path(str(record["path"])).with_name(f"{key}__frame_{frame_no:06d}.png"),
            width=int(record["width"]), height=int(record["height"]),
            objects=objects, video=key, frame_index=frame_no,
        ))

annotations = AnnotationSet(schema=SCHEMA, frames=tuple(frames), categories=(CLASS_NAME,))
print(f"{len(annotations.frames)} frames, "
      f"{sum(len(f.objects) for f in annotations.frames):,} point instances")

In [ ]:
STAGE    = DATASET / "_stage_images"
POLO_DIR = DATASET / "pose_dataset"
shutil.rmtree(STAGE, ignore_errors=True)
shutil.rmtree(POLO_DIR, ignore_errors=True)
STAGE.mkdir(parents=True)

# Symlinks, not copies: the renamed name is all that is wanted, and the pixels are
# already on this disk.
staged: list[AnnotationFrame] = []
for f in annotations.frames:
    source = f.image_path.with_name(f"frame_{f.frame_index:06d}.png")
    link = STAGE / f.image_path.name
    if not link.exists():
        try:
            link.symlink_to(source)
        except OSError:
            shutil.copy2(source, link)
    staged.append(AnnotationFrame(
        image_path=link, width=f.width, height=f.height,
        objects=f.objects, video=f.video, frame_index=f.frame_index,
    ))
staged_set = annotations.with_frames(tuple(staged))

# Group-aware by recording: every frame of the held-out trial goes to valid, so the
# score cannot be inflated by near-duplicate frames of the same fish on both sides.
split_of = {f.image_path.name: ("valid" if f.video.startswith(VALID_GROUP) else "train")
            for f in staged}

def polo_lines(frame: AnnotationFrame) -> list[str]:
    """One `<class> <radius> <x_rel> <y_rel>` row per fish."""
    lines = []
    for obj in frame.objects:
        kp = obj.keypoints[0]
        if not (math.isfinite(kp.x) and math.isfinite(kp.y)):
            continue
        # Clamp rather than let it through: Ultralytics discards a whole image as
        # corrupt when a coordinate falls outside [0, 1], so one stray fish at the
        # tank wall would cost the frame and every other fish in it.
        x = min(max(kp.x / frame.width, 0.0), 1.0)
        y = min(max(kp.y / frame.height, 0.0), 1.0)
        lines.append(format_polo_label_line(0, RADIUS_PX, x, y))
    return lines

written, skipped = write_split_tree(
    usable_frames(staged_set), POLO_DIR, split_of, polo_lines, symlink_images=True,
)
shutil.rmtree(STAGE, ignore_errors=True)

data_yaml = Path(make_polo_data_yaml(POLO_DIR, {CLASS_NAME: 0}, {0: RADIUS_PX}))
print(f"wrote {written} images, skipped {skipped}")
print(data_yaml.read_text())

### Check the labels before training

In [ ]:
fish_of = {make_entry_key(g, s): ds.load_tracks(g, s, run_id=VARIANT)["id"].nunique()
           for g, s in ENTRIES}

counts = {}
for split in ("train", "valid"):
    labels = sorted((POLO_DIR / split / "labels").glob("*.txt"))
    images = sorted((POLO_DIR / split / "images").glob("*.png"))
    counts[split] = (len(images), len(labels))
    for path in labels:
        for line in path.read_text().splitlines():
            fields = line.split()
            assert len(fields) == 4, f"{path.name}: expected 4 fields, got {len(fields)}"
            x, y = float(fields[2]), float(fields[3])
            assert 0.0 <= x <= 1.0 and 0.0 <= y <= 1.0, f"{path.name}: {x},{y} out of range"
        stem = path.stem.rsplit("__frame", 1)[0]
        n = len(path.read_text().splitlines())
        assert n == fish_of[stem], f"{path.name}: {n} points, expected {fish_of[stem]}"

for split, (n_img, n_lab) in counts.items():
    print(f"{split:6s} {n_img:4d} images  {n_lab:4d} labels")
assert all(n > 0 for pair in counts.values() for n in pair), "an empty split"
print("\nall labels parse, all coordinates in range, all counts match the tables")

The checks above cannot catch a flipped or transposed coordinate system -- every number
would still be in range. Drawing them on a real frame is what catches that.

In [ ]:
sample = sorted((POLO_DIR / "train" / "images").glob("*.png"))[0]
label = POLO_DIR / "train" / "labels" / f"{sample.stem}.txt"
image = plt.imread(sample)
h, w = image.shape[:2]

fig, ax = plt.subplots(figsize=(11, 6))
ax.imshow(image, cmap="gray")
for line in label.read_text().splitlines():
    _, radius, x, y = line.split()
    ax.add_patch(plt.Circle((float(x) * w, float(y) * h), float(radius),
                            fill=False, color="red", linewidth=1.4))
ax.set_title(f"{sample.name} -- {len(label.read_text().splitlines())} points, r={RADIUS_PX}")
ax.axis("off")
plt.tight_layout()
plt.show()

## 3 -- Train the point model

**Why no weights ship with this dataset.** The trainer, Ultralytics, is AGPL-3.0. Rather
than work through what distributing a model produced by it requires of the distributor,
this example carries none and shows you how to make your own. Everything else here runs
without it, because route B needs no model at all.

POLO ships under the distribution name `ultralytics`, so a POLO environment and an upstream
Ultralytics one are mutually exclusive -- one environment holds the fork or the release,
never both. The cell asks the *ladder* whether a POLO environment resolves and prints the
handoff rather than failing when it does not: same dataset on disk, different interpreter.

It never imports `ultralytics` into this kernel. That is not fastidiousness -- Ultralytics
is AGPL-3.0, and running it as a separate program is what keeps a mosaic install free of
it. `mosaic run --kind train-points` is that separate program.

In [ ]:
# POLO is AGPL-3.0 and runs as a separate program, so this asks the *environment*
# whether it resolves -- it never imports ultralytics into this kernel. Importing
# it here would make this notebook one work with it, which is exactly the
# separation `mosaic run --kind train-points` exists to keep.
from mosaic.tracking.common.toolenv import ToolNotFoundError, tool_invocation
from mosaic.tracking.common.ultralytics_env import POLO_ENV


def polo_invocation():
    """The argv that would launch POLO's `yolo`, or None. Runs nothing."""
    try:
        return tool_invocation(
            POLO_ENV, executable="yolo", conda_env=POLO_CONDA_ENV or None
        )
    except ToolNotFoundError:
        return None


TRAIN_PARAMS = {
    "data": str(data_yaml), "model": BASE_MODEL, "epochs": EPOCHS,
    "imgsz": IMGSZ, "batch": BATCH, "device": DEVICE, "patience": 50,
}

train_run_id = None
polo_argv = polo_invocation()
if not TRAIN:
    print("TRAIN is False -- not training.")
    print("Route A in section 4 needs a model: set TRAIN=True with a POLO environment,")
    print("or point POLO_MODEL at a best.pt you already have. Route B needs neither.")
    print(f"POLO environment: {'resolves' if polo_argv else 'not found'}")
elif polo_argv is None:
    print("No POLO environment. Build one (see docs/installation.md) and run:\n")
    print(f"  cd {DATASET}")
    print("  mosaic run -m dataset.yaml --kind train-points --params '"
          + json.dumps(TRAIN_PARAMS) + "'")
    print("\nThen re-run this notebook: section 4 picks the run up from models/.")
else:
    train_run_id = run_op(ds, "train-points", TRAIN_PARAMS)
    print("trained:", train_run_id)


`--overwrite` is **refused** with `--kind`, not ignored: an op decides reuse from its own
markers, and `"overwrite": true` goes inside `--params` when you really do mean it.

## 4 -- Track with TREx, two ways

TREx runs as a separate program in its own environment, found by the same five-step ladder
every external tool uses. Probe it *before* spending a conversion: the not-found error is
raised lazily inside the first entry's convert phase, by which point a run root, a tracks
variant and a failed run-log have already been written.

In [ ]:
# TREx is located by the ladder in section 0. Nothing is hard-coded here: setting
# MOSAIC_TREX_CONDA_ENV for a reader who did not ask for it would override their
# own $PATH trex silently, because the conda rung is tried first.
if TREX_CONDA_ENV:
    os.environ["MOSAIC_TREX_CONDA_ENV"] = TREX_CONDA_ENV
# os.environ["MOSAIC_TREX_BIN"] = "/abs/path/to/trex"
# os.environ["MOSAIC_TREX_DISPLAY"] = ":99"       # headless Linux needs an Xvfb

from mosaic.tracking.trex.run import TRexNotFoundError, _trex_invocation

# Resolving is not the same as existing. `conda run -n <env>` raises only when
# *conda* is missing, so a named environment that does not exist still returns an
# argv -- and the failure then surfaces deep inside the first entry's convert
# phase, after a run root and a failed run-log have been written. When the ladder
# hands back an absolute path, stat it; a bare name is unproven either way.
# Do NOT check by running `trex -h`: on macOS that opens a window and blocks.
try:
    TREX_ARGV = _trex_invocation()      # honours the full ladder; runs nothing
    target = Path(str(TREX_ARGV[-1]))
    if target.is_absolute() and not target.exists():
        raise TRexNotFoundError(
            f"the ladder resolved to {target}, which does not exist. "
            "Check MOSAIC_TREX_CONDA_ENV / MOSAIC_TREX_BIN."
        )
    print("TREx:", " ".join(str(a) for a in TREX_ARGV))
    if not target.is_absolute():
        print(f"  ('{target}' is a bare name -- resolved by the environment at run time,"
              "\n   so this probe cannot confirm it exists)")
except TRexNotFoundError as exc:
    TREX_ARGV = None
    print(f"TREx unavailable -- sections 4-7 are skipped.\n  {exc}")


### Where the outline comes from

Two mechanisms, with confusingly similar names:

| | what it is | needs a model? |
|---|---|---|
| `detect_type` | the **conversion-time** method that separates foreground from background. One of `none`, `yolo`, `sam3`, `background_subtraction`, `precomputed` | `yolo` does; **`background_subtraction` does not** |
| `track_background_subtraction` | a **tracking-time** flag: contrast a blob against the background before thresholding it | no -- it refines whatever is already in the `.pv` |

**Route A** is `detect_type="yolo"` with the point model. A point detector supplies no
outline, so TREx wraps each detection in a **disc** of `detect_point_radii` -- default
20 px, a `{class: radius}` map, and *not* read from the model. A disc has no long axis, but
it holds the image pixels, so `track_background_subtraction` together with
`track_posture_threshold` recovers the real outline inside it. Set neither and the midline
is fitted to a circle, which is why `POSTURE_THRESHOLD` is not optional.

Two notes on that pair. The threshold is a property of *your* footage -- contrast,
illumination, animal size -- and a poorly chosen one produces an angle that varies on every
frame and means nothing, so it is worth checking on a short clip before a long run. And
`detect_point_radii` is **absent from TREx's published parameter list**; it is registered in
the source (`default_config.cpp`, "An array of radii for a given point class in a POLO
network") and defaults to 20 when unset.

**Route B** is `detect_type="background_subtraction"` with no model: subtract a background
estimate, threshold, and the blob *is* the animal. Posture works natively -- no radius, no
posture threshold. What it needs instead is contrast and a static background, plus a
background estimate worth the name: `averaging_method="mode"` is a per-pixel median, and
because the background is sampled from *inside* the converted range, a short range is what
puts a slow animal into the background.

One more, common to both: `video_conversion_range` bounds the **conversion**.
`analysis_range` does not -- it bounds tracking, by which point every frame has already been
through the detector.

In [ ]:
# convert_extra_settings is a passthrough to TREx's own settings.
convert_extra = ({"video_conversion_range": [0, int(CONVERT_FRAMES)]}
                 if CONVERT_FRAMES else None)

def model_reference() -> Optional[str]:
    """A weights path or a registered train-points run id, or None.

    POLO_MODEL wins, then whatever section 3 trained, then any train-points run
    already in this dataset -- so a reader who trained on a previous pass picks it
    up without editing anything.
    """
    if POLO_MODEL is not None:
        assert Path(POLO_MODEL).exists(), f"POLO_MODEL names nothing: {POLO_MODEL}"
        return str(POLO_MODEL)
    if train_run_id is not None:
        return train_run_id
    trained = sorted((ds.get_root("models") / "train-points").glob("train-points.*"))
    for run in trained:
        if (run / "train" / "weights" / "best.pt").exists():
            return run.name
    return None

MODEL_REF = model_reference()
print(f"route A model: {MODEL_REF or 'none -- route A will be skipped'}")

In [ ]:
# One dict per entry, so sections 5-7 iterate whatever ran -- one route or two.
VARIANTS: dict[tuple[str, str], dict[str, str]] = {}

if TREX_ARGV is not None:
    for group, sequence in TRACK_ENTRIES:
        # From the table, not the directory name: "70-fish" is a label, and the count
        # that matters is the one the data actually holds.
        n_fish = int(ds.load_tracks(group, sequence, run_id=VARIANT)["id"].nunique())
        produced: dict[str, str] = {}
        print(f"-- {group}/{sequence}: {n_fish} individuals")

        if MODEL_REF is not None:
            produced["A: POLO point model"] = run_op(ds, "trex", {
                "entries": [f"{group}:{sequence}"],       # "group:sequence" tokens
                "detect_model": MODEL_REF,                # run id or a bare .pt path
                "detect_type": "yolo",                    # POLO loads through this path
                "detect_conf_threshold": DETECT_CONF,
                "track_max_individuals": n_fish,
                "auto_train": False,                      # golden shiners look alike;
                                                          # visual ID has nothing to learn
                "convert_extra_settings": convert_extra,
                "track_extra_settings": {
                    "track_background_subtraction": BACKGROUND_SUBTRACTION,
                    "track_posture_threshold": POSTURE_THRESHOLD,
                },
                "idle_timeout": 1800,                     # HASH_EXCLUDE: not in the run_id
            })

        produced["B: model-free subtraction"] = run_op(ds, "trex", {
            "entries": [f"{group}:{sequence}"],
            "detect_model": None,                         # <- no model at all
            "detect_type": "background_subtraction",
            "track_max_individuals": n_fish,
            "auto_train": False,
            "convert_extra_settings": {
                **(convert_extra or {}),
                "detect_threshold": BGSUB_DETECT_THRESHOLD,
                "averaging_method": BGSUB_AVERAGING,
                "average_samples": BGSUB_AVERAGE_SAMPLES,
            },
            "idle_timeout": 1800,
        })

        VARIANTS[(group, sequence)] = produced
        for label, run_id in produced.items():
            print(f"   {label:28s} {run_id}")
    if MODEL_REF is None:
        print("\nRoute A skipped: no model. See section 3.")
else:
    print("skipped -- no TREx")

## 5 -- What each route produced

In [ ]:
idx = read_tracks_index(ds)
print(idx[["group", "sequence", "run_id", "producer", "n_rows", "n_keypoints"]]
      .to_string(index=False))

Every recipe now sits in one index. That is also why every read below passes `run_id=`
explicitly: an entry carrying more than one variant is a question mosaic **refuses to
guess at**, so an unqualified `load_tracks` raises rather than silently picking one.

In [ ]:
if not VARIANTS:
    print("Nothing tracked -- see section 4. (No TREx, or no route ran.)")

for (group, sequence), produced in VARIANTS.items():
    print(f"{group}/{sequence}")
    for label, run_id in produced.items():
        table = ds.load_tracks(group, sequence, run_id=run_id)
        span = f"{int(table['frame'].min())}-{int(table['frame'].max())}"
        print(f"  {label:28s} {len(table):8,} rows  {table['id'].nunique():3d} ids  "
              f"frames {span}")
    print()

### What each route calls `X`/`Y`

Worth checking before anything downstream reads these tables.

The pseudo-annotations in section 2 came from the published `X`/`Y`, and in that format the
column is the **head**. So route A's model detects heads, its disc is centred on a head, and
the centroid TREx computes for a head-centred disc is still the head. Route B segments the
whole fish, so its centroid is a real body centre.

TREx reports both a centre and a `#head`, so each variant can be asked how far apart they
are -- no external reference needed.

In [ ]:
if not VARIANTS:
    print("Nothing tracked -- see section 4. (No TREx, or no route ran.)")

for (group, sequence), produced in VARIANTS.items():
    fish_px = float(ds.load_tracks(group, sequence, run_id=VARIANT)["body_length"].median()) / 1.5
    print(f"{group}/{sequence} -- fish ~{fish_px:.0f} px long")
    tables = {}
    for label, run_id in produced.items():
        t = ds.load_tracks(group, sequence, run_id=run_id)
        tables[label] = t
        if {"X#head", "Y#head"} <= set(t.columns):
            sep = float(np.nanmedian(np.hypot(t["X"] - t["X#head"], t["Y"] - t["Y#head"])))
            print(f"  {label:28s} X/Y sits {sep:5.1f} px "
                  f"({sep / fish_px:.0%} of a body length) from its own X#head")

    if len(tables) == 2:
        # Nearest-neighbour, not by id: two tracker runs number individuals
        # independently, so only position can be compared.
        (la, ta), (lb, tb) = tables.items()
        offsets = []
        for frame in range(100, min(int(ta["frame"].max()), int(tb["frame"].max())), 100):
            a = ta[ta["frame"] == frame][["X", "Y"]].to_numpy(dtype=float)
            b = tb[tb["frame"] == frame][["X", "Y"]].to_numpy(dtype=float)
            a, b = a[np.isfinite(a).all(axis=1)], b[np.isfinite(b).all(axis=1)]
            if not len(a) or not len(b):
                continue
            d = np.hypot(a[:, None, 0] - b[None, :, 0], a[:, None, 1] - b[None, :, 1])
            offsets.append(float(np.median(d.min(axis=1))))
        if offsets:
            print(f"  the two routes' X/Y differ by {np.median(offsets):.1f} px")
    print()

Read the numbers the cell above printed, not the ones in this paragraph -- on the
default settings only route B runs, so only one line appears.

**Route B** segments the whole fish, so its centroid is a real body centre and sits
about a fifth of a body length from its own reported head. That is what a genuine
head-to-centroid distance looks like.

**Route A**, if you trained a model and it ran, sits about a *pixel* from its own
reported head. A symmetric disc has no end to call the head, so the two collapse
together -- the detection was centred on a head, and the centroid of a head-centred
disc is still a head.

**That difference matters more than it looks.** mosaic's track schema says `X`/`Y` is
the **body centre** on every tracker -- that invariant is what lets a feature compare
across trackers at all. Route A satisfies it in name only: its `X`/`Y` is a head, and
nothing on disk says so. Feed both variants to a social-distance feature and you get
two answers, one of them off by a fifth of a body length, with no error anywhere.

Three ways out, in order of effort:

1. **Use route B**, whose `X`/`Y` is already a body centre.
2. **Train the point model on the body centre instead of the head.** This format does
   not carry one, but it carries `ANGLE` and `body_length`, so a centre can be derived
   and the emitter in section 2 pointed at that instead. The model then detects what
   the schema promises.
3. **Keep the head and say so** -- but then the variant should not claim `mosaic_v1`
   semantics, and every downstream comparison needs the offset applied by hand.


## 6 -- Tracked frames from across the recording

In [ ]:
from mosaic.behavior.visualization_library.playback import build_overlay
from mosaic.behavior.visualization_library.video_stream import render_stream

N_STILLS = 5

if not VARIANTS:
    print("Nothing tracked -- see section 4. (No TREx, or no route ran.)")

for (group, sequence), produced in VARIANTS.items():
    resolved = ds.resolve_media(group, sequence)
    for label, run_id in produced.items():
        overlay_data, tracks_df, _ = build_overlay(
            ds, group=group, sequence=sequence,
            feature_runs={}, label_kind=None,   # no converted behaviour labels here
            tracks_run_id=run_id,
        )
        picks = np.linspace(0, int(tracks_df["frame"].max()), N_STILLS, dtype=int).tolist()

        fig, axes = plt.subplots(1, N_STILLS, figsize=(4.6 * N_STILLS, 3.0))
        for ax, k in zip(np.atleast_1d(axes), picks):
            # start == end yields exactly one frame; the stream seeks rather than scanning.
            stream = render_stream(resolved.paths, overlay_data,
                                   start=int(k), end=int(k), facts=resolved.facts)
            frame_idx, frame = next(iter(stream))
            stream.close()
            ax.imshow(frame[:, :, ::-1])          # BGR -> RGB
            ax.set_title(f"frame {frame_idx}")
            ax.axis("off")
        fig.suptitle(f"{group}/{sequence} -- {label}", y=1.04)
        plt.tight_layout()
        plt.show()

## 7 -- A 30-second annotated video per route

`overlay` is a registered feature, not a helper -- so the rendered video is addressed by
`run_id` like every other artifact, and a graph can end on the deliverable rather than one
step short of it.

The overlay is written with OpenCV's `mp4v`, which most browsers will not play, so one
ffmpeg pass to H.264 makes it embeddable.

In [ ]:
from mosaic.behavior.visualization_library import Overlay
from IPython.display import Video, display

CLIP_SECONDS   = 30
EMBED_LIMIT_MB = 8      # embedding base64-encodes the clip into the .ipynb

if not VARIANTS:
    print("Nothing tracked -- see section 4. (No TREx, or no route ran.)")

for (group, sequence), produced in VARIANTS.items():
    fps = float(media[(media["group"] == group)
                      & (media["sequence"] == sequence)].iloc[0]["fps"])
    for label, run_id in produced.items():
        last = int(ds.load_tracks(group, sequence, run_id=run_id)["frame"].max())
        result = ds.run_feature(
            Overlay(params={
                "label_kind": None,   # the default is "behavior"; without labels that
                                      # only prints a warning per entry
                "start": 0,
                "end": min(int(round(fps * CLIP_SECONDS)) - 1, last),
                "downscale": 0.5,     # 1920x1032 is more than a notebook needs
                "draw_options": {"point_radius": 4, "bbox_thickness": 2},
            }),
            entries=[(group, sequence)],
            tracks_run_id=run_id,
        )
        raw = (feature_run_root(ds, result.feature, result.run_id)
               / f"{make_entry_key(group, sequence)}.mp4")
        h264 = raw.with_name("overlay_h264.mp4")
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw),
             "-c:v", "libx264", "-crf", "20", "-pix_fmt", "yuv420p", str(h264)],
            check=True,
        )
        size_mb = h264.stat().st_size / 1e6
        print(f"{group}/{sequence} -- {label}\n  {h264}  ({size_mb:.1f} MB)")
        if size_mb <= EMBED_LIMIT_MB:
            display(Video(str(h264), embed=True, width=720))
        else:
            print(f"  not embedding ({size_mb:.0f} MB > {EMBED_LIMIT_MB} MB); open the "
                  "path above, or lower CLIP_SECONDS / downscale")

## 8 -- Where to go from here

What this notebook covers, in order:

1. Published tracks make serviceable **pseudo-annotations** -- no manual labelling at all.
2. A POLO point model trains on 160 frames from two densities and is validated on
   80 frames from a third density it never saw.
3. TREx tracks with it, recovering a body axis from the pixels inside each detection disc
   -- which is what `track_background_subtraction` and `track_posture_threshold` are for.
4. TREx also tracks the same footage **with no model at all** -- which is what runs on
   the default settings. Train a model and section 5 shows the two routes disagreeing
   about what `X`/`Y` means.

**Neither route is presented as better.** They have different prerequisites: route B needs
contrast and a static background and nothing else; route A needs annotations, a training
environment and a GPU, and earns that where route B's assumptions fail -- changing
illumination, a textured or moving background, animals that touch, debris the size of an
animal, several species to tell apart. Which one suits your footage is an empirical
question, and answering it properly means a parameter study rather than a single run of
this notebook.

Natural next steps: run route B first on your own recordings, since it is nearly free, and
use what it gets wrong to decide whether a detector is worth training; hold out a whole
recording and score detection against the published points; or feed either tracked variant
into the collective-motion features, which take position, identity and heading.

Every variant is addressed by `run_id`, so re-running any cell above is a cache hit until a
parameter actually changes.